[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_3_camera_model/07_camera_calibration.ipynb)

# Notebook 07 — Camera Calibration
### 6D Pose Vision Workshop · Part 3: Camera Model

**Estimated time:** 50 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video notes 3 (*Camera Calibration Explained*), 4 (*OpenCV Python Camera Calibration*), 15 (*Calibration in < 5 Minutes*)

---

## Recommended Watch

> Watch **both** before opening the notebook — the first gives a quick workflow overview, the second goes deeper into K and distortion coefficients.

| # | Title | Link | Duration |
|---|---|---|---|
| 1 | **Camera Calibration in less than 5 Minutes with OpenCV** | [▶ Watch](https://www.youtube.com/watch?v=_-BTKiamRTg) | ~5 min |
| 2 | **OpenCV Python Camera Calibration (Intrinsic, Extrinsic, Distortion)** | [▶ Watch](https://www.youtube.com/watch?v=H5qbRTikxI4) | ~20 min |

---

> **Optional — after finishing this notebook:**

> **ChArUco boards** combine a chessboard pattern with ArUco markers — more accurate and robust than a plain chessboard. Worth knowing once you're comfortable with the standard workflow.

| # | Title | Link | Duration |
|---|---|---|---|
| Optional | **Camera Calibration with ChArUco Boards (OpenCV)** | [▶ Watch](https://youtu.be/EUvco3rjUdQ?si=f92x9E5rnwwZgZtH) | — |

> **Optional — want to understand what `cv2.calibrateCamera` is doing under the hood?**
> Dr. Shree K. Nayar (Columbia University) walks through the math: how 3D-2D correspondences are used to build a linear system, why it becomes an eigenvalue problem, and how P = K[R|t] is recovered.

| # | Title | Link |
|---|---|---|
| Optional | **Camera Calibration — Estimating the Projection Matrix** *(VN35 — Dr. Nayar)* | [▶ Watch](https://youtu.be/GUbWsXU1mac?si=gEeYXRU7Tkynk-mY) |
| Optional | **Camera Calibration \| Uncalibrated Stereo** *(playlist — Dr. Nayar)* | [▶ Playlist](https://youtube.com/playlist?list=PL2zRqk16wsdoCCLpou-dGo7QQNks1Ppzo&si=SesSky5u4rSbtuKg) |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, json, glob

print(f"OpenCV: {cv2.__version__}")

def show_images(pairs, figsize_per=(5, 4)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

## Learning Objectives

By the end of this notebook you will:

- Understand **why** calibration is necessary and what happens without it
- Know how to **generate and capture** chessboard calibration images
- Use `cv2.findChessboardCorners` and `cv2.cornerSubPix` to detect corners
- Run `cv2.calibrateCamera` to compute $K$ and distortion coefficients
- Interpret the **reprojection error** as a quality metric
- **Save and load** calibration data to reuse across sessions

---

## 1. Why Calibration Is Critical

### What is calibration, really?

Think of it like **putting on glasses for the first time**.

When you hold a ruler at arm's length, your brain automatically knows how far away it is and how long it is — because it has been "calibrated" by years of experience.

<img src="https://lh3.googleusercontent.com/d/12W5B35F66TB8FFjHEw14XD8C7s0zJdNO" width="700" alt="Brain calibration analogy — ruler at arm's length"/>

A camera has no such built-in knowledge. It only sees pixels. Calibration is the process of teaching it the relationship between what it sees in pixels and what is actually out there in the 3D world (in centimeters or meters).

<img src="https://lh3.googleusercontent.com/d/1k1RTJ_gOF8pvcwfr-_L_nTUGCK7e8-nF" width="700" alt="Camera sees only pixels — needs calibration to understand the world"/>

More precisely: calibration means finding the **intrinsic matrix K** — the four numbers (fx, fy, cx, cy) that describe *your specific camera's* lens and sensor. Every camera model is slightly different, so K must be measured for each individual camera.

<img src="https://lh3.googleusercontent.com/d/1MXKjHoASeL7BavaTXoOXEUWUUHGwzzHB" width="700" alt="Intrinsic matrix K maps pixel distances to real-world angles and distances"/>

Without knowing K, you cannot answer questions like:
- "That pixel is 200 pixels from center — how many degrees off-axis is that object?"
- "The object appears 100 pixels tall — how far away is it?"

### Without calibration you cannot:

- Accurately estimate object pose *(in NB09 you'll use `solvePnP` to recover the 3D pose of an object — it requires K to be accurate, otherwise every result will be wrong)*
- Measure real-world distances from images
- Correctly overlay AR graphics (they'll drift as the camera moves)
- Do reliable stereo depth estimation

### The chessboard method — why it works

The chessboard has corners at **known 3D positions** (on a flat plane at Z=0). When we photograph the chessboard at different angles:

1. We know where each corner is in the world: $(0,0,0),\,(1,0,0),\,(2,0,0),\,\ldots$ (in square units)
2. We detect where each corner appears in the image (pixels)
3. We have **known (world point → image point) pairs** for many views
4. `cv2.calibrateCamera` solves a system of equations to find K and distortion that best explain all these correspondences

The more views, and the more varied the angles, the better the result.

<img src="https://lh3.googleusercontent.com/d/1ksXBJUyEI2f8FA3qN6FA2QObxSckzoKu" width="700" alt="Chessboard corners at known 3D positions photographed from multiple angles"/>

### Critical counting rule

A 9×6 chessboard refers to **inner corners only** — not the number of squares:

```
  9×6 inner corners:
  ┼─┼─┼─┼─┼─┼─┼─┼─┼
  ├─┼─┼─┼─┼─┼─┼─┼─┤
  ├─┼─┼─┼─┼─┼─┼─┼─┤
  ├─┼─┼─┼─┼─┼─┼─┼─┤
  ├─┼─┼─┼─┼─┼─┼─┼─┤
  └─┴─┴─┴─┴─┴─┴─┴─┘
  ← 9 columns of corners, 6 rows = 54 total inner corners
  (a board with 9×6 inner corners has 10×7 squares!)
```

Miscounting = calibration failure (0 corners detected).

---

In [ ]:
# Generate a synthetic chessboard + calibration images for this notebook
# (In a real workflow: print the chessboard and capture photos)

os.makedirs('../assets/calibration/chessboard_images', exist_ok=True)
os.makedirs('../assets/calibration', exist_ok=True)

def create_chessboard_image(rows=7, cols=10, square_size=60, border=40):
    """Generate a printable chessboard pattern.
    
    For CHESSBOARD_SIZE = (9, 6) inner corners you need 10 cols x 7 rows of squares.
    Inner corners = (cols-1) x (rows-1).
    """
    h = rows * square_size + 2 * border
    w = cols * square_size + 2 * border
    img = np.ones((h, w), dtype=np.uint8) * 255
    for r in range(rows):
        for c in range(cols):
            if (r + c) % 2 == 0:
                y1 = border + r * square_size
                x1 = border + c * square_size
                img[y1:y1+square_size, x1:x1+square_size] = 0
    return img

chessboard = create_chessboard_image()
cv2.imwrite('../assets/calibration/chessboard_9x6.png', chessboard)

plt.figure(figsize=(10, 6))
plt.imshow(chessboard, cmap='gray')
plt.title('Generated chessboard for CHESSBOARD_SIZE=(9,6)\n'
          '10 cols × 7 rows of squares → 9×6 = 54 inner corners')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f"Saved: ../assets/calibration/chessboard_9x6.png")

In [ ]:
# Generate synthetic calibration images (simulating holding chessboard at different angles)
# In real use: replace with photos from your actual camera

def make_synthetic_calib_image(board, K_true, dist_true, rvec, tvec, img_size=(640, 480)):
    """Project board corners through camera model to generate a synthetic calibration image."""
    rows, cols = 6, 9  # inner corners
    
    # Build object points (inner corners, Z=0)
    obj_pts = np.zeros((rows * cols, 3), np.float32)
    obj_pts[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    obj_pts *= 0.03  # 3cm squares in world units (meters)
    
    # Project corners
    img_pts, _ = cv2.projectPoints(obj_pts, rvec, tvec, K_true, dist_true)
    img_pts = img_pts.reshape(-1, 2)
    
    # Draw a synthetic image
    img = np.full((*img_size[::-1], 3), 200, dtype=np.uint8)
    
    # Create board image: needs (rows+1) x (cols+1) squares to produce (rows x cols) inner corners
    board_gray = create_chessboard_image(rows+1, cols+1, square_size=30, border=20)
    board_h, board_w = board_gray.shape
    
    # Source corners of the board image (inner area, excluding border)
    src = np.array([[20, 20],
                    [board_w-20, 20],
                    [board_w-20, board_h-20],
                    [20, board_h-20]], dtype=np.float32)
    
    # Destination: project 4 outer corners (one square outside the inner corner grid)
    outer_obj = np.array([[-1, -1, 0], [cols, -1, 0],
                           [cols, rows, 0], [-1, rows, 0]], dtype=np.float32) * 0.03
    outer_img, _ = cv2.projectPoints(outer_obj, rvec, tvec, K_true, dist_true)
    dst = outer_img.reshape(-1, 2).astype(np.float32)
    
    M = cv2.getPerspectiveTransform(src, dst)
    board_rgb = cv2.cvtColor(board_gray, cv2.COLOR_GRAY2BGR)
    warped = cv2.warpPerspective(board_rgb, M, (img_size[0], img_size[1]))
    
    mask = cv2.warpPerspective(
        np.ones((board_h, board_w), dtype=np.uint8) * 255, M,
        (img_size[0], img_size[1]))
    img[mask > 0] = warped[mask > 0]
    
    # Add slight noise
    noise = np.random.randint(-8, 8, img.shape, dtype=np.int8)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

# True camera parameters (what we'll try to recover)
K_true = np.array([[720, 0, 320],
                   [0, 720, 240],
                   [0,   0,   1]], dtype=np.float64)
dist_true = np.array([-0.3, 0.1, 0.001, 0.002, -0.05])  # radial + tangential

# Generate 12 images with varied poses
poses = [
    (np.array([[ 0.0,  0.0,  0.1]]), np.array([[0.0,  0.0,  0.8]])),
    (np.array([[ 0.3,  0.0,  0.0]]), np.array([[0.0,  0.0,  0.9]])),
    (np.array([[-0.3,  0.0,  0.0]]), np.array([[0.0,  0.0,  0.85]])),
    (np.array([[ 0.0,  0.3,  0.0]]), np.array([[0.0,  0.0,  0.9]])),
    (np.array([[ 0.0, -0.3,  0.0]]), np.array([[0.0,  0.0,  0.85]])),
    (np.array([[ 0.4,  0.3,  0.2]]), np.array([[0.05, 0.0,  0.95]])),
    (np.array([[-0.4, -0.3,  0.0]]), np.array([[-0.05,0.0,  0.90]])),
    (np.array([[ 0.2, -0.4,  0.1]]), np.array([[0.0,  0.05, 0.80]])),
    (np.array([[-0.2,  0.4, -0.1]]), np.array([[0.0, -0.05, 0.85]])),
    (np.array([[ 0.5,  0.0,  0.3]]), np.array([[0.1,  0.0,  1.00]])),
    (np.array([[-0.5,  0.0, -0.2]]), np.array([[-0.1, 0.0,  0.95]])),
    (np.array([[ 0.0,  0.5,  0.4]]), np.array([[0.0,  0.1,  0.90]])),
]

calib_imgs = []
for i, (rvec, tvec) in enumerate(poses):
    try:
        img = make_synthetic_calib_image(
            create_chessboard_image(), K_true, dist_true, rvec, tvec)
        path = f'../assets/calibration/chessboard_images/calib_{i:02d}.jpg'
        cv2.imwrite(path, img)
        calib_imgs.append(img)
    except Exception as e:
        print(f"  Skipped pose {i}: {e}")

print(f"Generated {len(calib_imgs)} synthetic calibration images")
# Show a sample
show_images([(calib_imgs[i], f'Calib image {i}') for i in [0, 5, 10]])

## 2. Detecting Chessboard Corners

### Step 1: findChessboardCorners

Finds the inner corner grid automatically:

```python
ret, corners = cv2.findChessboardCorners(gray, (cols, rows), flags)
# cols, rows = inner corners (NOT squares!) — e.g., (9, 6)
# ret = True if corners found, False otherwise
# corners = array of shape (N, 1, 2) — N = cols * rows
```

### Step 2: cornerSubPix — subpixel refinement

The coarse detection is accurate to ~1 pixel. `cornerSubPix` refines to **sub-pixel accuracy** by iteratively finding the exact local maximum of the corner response:

```python
corners_refined = cv2.cornerSubPix(gray, corners,
    winSize=(11, 11),   # search window size around each corner
    zeroZone=(-1, -1),  # -1 = no zero zone
    criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
)
```

Subpixel accuracy is critical for calibration — a 0.5px error in corner location translates directly into calibration error.

In [ ]:
# Detect corners in all calibration images

CHESSBOARD_SIZE = (9, 6)  # (cols, rows) inner corners — MUST MATCH YOUR BOARD

# Termination criteria for cornerSubPix
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Build the object points: (0,0,0), (1,0,0), ..., (8,5,0) — in square units
# We multiply by square_size_meters later if we want real-world scale
SQUARE_SIZE = 0.03  # 3 cm squares (adjust to your actual board)

objp = np.zeros((CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE  # convert to meters

print(f"Object point shape: {objp.shape}")
print(f"First 4 object points (meters):\n{objp[:4]}")
print(f"These define a grid on the Z=0 plane")

# Collect points from all images
objpoints = []  # 3D world points (same for every image)
imgpoints = []  # 2D image points (different per image)
good_images = []

image_paths = sorted(glob.glob('../assets/calibration/chessboard_images/calib_*.jpg'))

for path in image_paths:
    img = cv2.imread(path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Find corners
    ret, corners = cv2.findChessboardCorners(gray, CHESSBOARD_SIZE, None)
    
    if ret:
        # Refine to subpixel accuracy
        corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        
        objpoints.append(objp)
        imgpoints.append(corners_refined)
        good_images.append(img.copy())
        
        # Draw corners on image for visualization
        cv2.drawChessboardCorners(img, CHESSBOARD_SIZE, corners_refined, ret)

print(f"\nFound corners in {len(good_images)}/{len(image_paths)} images")

# Show detected corners on first few images
annotated = []
for i, path in enumerate(image_paths[:4]):
    img = cv2.imread(path)
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, CHESSBOARD_SIZE, None)
    if ret:
        corners_r = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        cv2.drawChessboardCorners(img, CHESSBOARD_SIZE, corners_r, ret)
    annotated.append((img, f'Image {i}: {"✓" if ret else "✗"} corners'))

if annotated:
    show_images(annotated)

## 3. Running Camera Calibration

### What does `calibrateCamera` actually do?

Imagine you're trying to figure out the rules of a mystery projector. You can't open it up — but you can show it known slides and measure where each dot lands on the screen. By comparing "where I expected the dot" vs "where the dot actually landed," you can reverse-engineer the projector's internal settings.

That's exactly what `cv2.calibrateCamera` does:

1. You give it the **3D positions of all corners** (you know these — it's your chessboard)
2. You give it the **detected pixel positions** of those same corners (from the previous step)
3. It tries different values of K and distortion, projects the 3D corners into the image using those values, and checks how close the projected positions are to what you actually detected
4. It tweaks K and distortion until the projected positions match the detected ones as closely as possible

The measure of "how close" is called **reprojection error** — more on that below.

---

### What about distortion?

Remember K from NB06? K assumes the camera is a **perfect pinhole** — light travels in a perfectly straight line through a single point, and hits the sensor exactly where the math says it should.

Real camera lenses are **not perfect pinholes**. Glass curves and bends light slightly — and the bending is different at different parts of the lens. This means pixels don't land exactly where the pinhole model predicts. That gap between "where the pinhole model says the pixel should be" and "where it actually lands" is called **distortion**.

There are two main types:

---

#### Radial distortion — the fisheye effect

This is the most common. Light rays that pass through the **outer edges** of a lens get bent more than rays that pass through the **center**. The result: straight lines in the real world appear curved in the image.

<img src="https://lh3.googleusercontent.com/d/11YCmPaWRinBVHkpdUUNahDe8gvL0iMyE" width="700" alt="Barrel distortion vs pincushion distortion vs no distortion"/>

Two flavors:
- **Barrel distortion** (k1 < 0): straight lines bow *outward* — common in wide-angle and webcam lenses
- **Pincushion distortion** (k1 > 0): straight lines bow *inward* — common in telephoto lenses

**How it's modeled:**

The distortion effect grows with distance from the image center. If a point is at distance $r$ from the center (in normalized coordinates), the correction factor is:

$$\text{factor} = 1 + k_1 r^2 + k_2 r^4 + k_3 r^6$$

And the distorted position becomes:

$$x_{\text{distorted}} = x \cdot \text{factor} \qquad y_{\text{distorted}} = y \cdot \text{factor}$$

**Intuition for the formula:**
- At the **center** of the image: $r = 0$, so factor $= 1$ → no distortion at all
- Further from center: $r$ grows, the polynomial kicks in, factor moves away from 1
- $k_1, k_2, k_3$ control *how fast* the distortion grows. Most cameras only need $k_1$ and $k_2$; $k_3$ is for extreme wide-angle lenses

Think of it as: *"multiply the ideal position by a correction factor that grows polynomially with distance from center."* The polynomial shape is smooth and can capture the gradual bending that real lenses produce.

---

#### Tangential distortion — the tilted lens effect

This happens when the lens is not perfectly parallel to the image sensor (a manufacturing imperfection). Instead of just scaling radially, points shift slightly *sideways*.

<img src="https://lh3.googleusercontent.com/d/1A3I54YRMpO89AAcY4pZ4LIrebZE0ncb9" width="700" alt="Tangential distortion — lens not parallel to sensor"/>

It's modeled by two coefficients $p_1$ and $p_2$:

$$x_{\text{distorted}} = x + \bigl[2p_1 xy + p_2(r^2 + 2x^2)\bigr]$$
$$y_{\text{distorted}} = y + \bigl[p_1(r^2 + 2y^2) + 2p_2 xy\bigr]$$

**Intuition:** The correction is *added* to the ideal position (not multiplied), and it involves cross terms like $xy$ — meaning the shift depends on *which direction* you are from the center, not just how far. In practice, $p_1$ and $p_2$ are very small numbers for most cameras (manufacturing is pretty good).

---

#### The full distortion vector

OpenCV stores all five distortion parameters together:

```python
dist = [k1, k2, p1, p2, k3]
#        ^   ^   ^   ^   ^
#        radial  tang  radial
```

That's it — **5 numbers** that describe how your specific lens bends light. Together with K (4 numbers), that's 9 numbers that fully characterize your camera.

**Key point:** `calibrateCamera` finds K *and* these 5 distortion coefficients simultaneously — because the chessboard images contain enough information to solve for all 9 at once. You don't have to do them in separate steps.

---

### cv2.calibrateCamera

```python
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objectPoints,  # list of (N,3) arrays — 3D corners for each image
    imagePoints,   # list of (N,1,2) arrays — 2D corners for each image
    imageSize,     # (width, height) of images
    None,          # initial K (None = auto-initialize)
    None           # initial distortion (None = auto-initialize)
)
```

**Returns:**
- `ret` — mean reprojection error (pixels) — your main quality metric
- `K` — 3×3 intrinsic matrix
- `dist` — distortion coefficients `[k1, k2, p1, p2, k3]`
- `rvecs` — list of rotation vectors (one per image)
- `tvecs` — list of translation vectors (one per image)

---

### Reprojection error — the quality metric

**In plain English:** after calibration, take each 3D corner point, project it through the estimated K back onto the image, and measure the distance (in pixels) from where it lands to where you actually detected it. If K is perfect, the two should coincide exactly. In practice they won't be identical — the average gap across all corners and all images is the reprojection error.

**Formally:**

$$\text{reprojection error} = \frac{1}{N} \sum_{i=1}^{N} \| \mathbf{x}_i^{\text{detected}} - \hat{\mathbf{x}}_i^{\text{projected}} \|_2$$

Where:
- $\mathbf{x}_i^{\text{detected}}$ = pixel location we actually detected
- $\hat{\mathbf{x}}_i^{\text{projected}}$ = pixel location predicted by our estimated K and distortion

| Error | Quality |
|---|---|
| < 0.5 px | Excellent |
| < 1.0 px | Good — usable for robotics |
| 1.0–2.0 px | Acceptable — may cause pose jitter |
| > 2.0 px | Poor — recapture images |

In [ ]:
# Run calibration!

if len(objpoints) < 6:
    print(f"WARNING: Only {len(objpoints)} images with corners detected.")
    print("Calibration needs at least 6–10 images for reliable results.")

# Get image size from first image
img_size = (cv2.imread(image_paths[0]).shape[1],   # width
            cv2.imread(image_paths[0]).shape[0])    # height

print(f"Running calibrateCamera with {len(objpoints)} images, image size {img_size}...")

ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, img_size, None, None
)

print(f"\nMean reprojection error: {ret:.4f} pixels")
quality = 'Excellent' if ret < 0.5 else 'Good' if ret < 1.0 else 'Acceptable' if ret < 2.0 else 'Poor'
print(f"Quality: {quality}")

print(f"\nCamera Matrix K:")
print(K)
print(f"\n  fx = {K[0,0]:.2f} px  (true: {K_true[0,0]})")
print(f"  fy = {K[1,1]:.2f} px  (true: {K_true[1,1]})")
print(f"  cx = {K[0,2]:.2f} px  (true: {K_true[0,2]})")
print(f"  cy = {K[1,2]:.2f} px  (true: {K_true[1,2]})")

print(f"\nDistortion coefficients [k1, k2, p1, p2, k3]:")
print(dist.flatten())
print(f"True: {dist_true}")

> **Curious what happens inside `cv2.calibrateCamera`?**
> Before running the cell below, watch [VN35 — Camera Calibration: Estimating the Projection Matrix](https://youtu.be/GUbWsXU1mac?si=gEeYXRU7Tkynk-mY) *(Dr. Nayar, ~15 min)*. It explains exactly how the 3D-2D correspondences you just collected are used to build a linear system, solve an eigenvalue problem, and recover P = K[R|t] — which is precisely what the next cell does.

In [ ]:
# Compute per-image reprojection error — useful to find bad calibration images

per_image_errors = []

for i, (objp_i, imgp_i, rvec, tvec) in enumerate(zip(objpoints, imgpoints, rvecs, tvecs)):
    # Reproject 3D points back to 2D using estimated parameters
    reprojected, _ = cv2.projectPoints(objp_i, rvec, tvec, K, dist)
    
    # Compute pixel-wise error
    error = cv2.norm(imgp_i, reprojected, cv2.NORM_L2) / len(reprojected)
    per_image_errors.append(error)

# Plot per-image errors
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['green' if e < 0.5 else 'orange' if e < 1.0 else 'red'
          for e in per_image_errors]
axes[0].bar(range(len(per_image_errors)), per_image_errors, color=colors)
axes[0].axhline(0.5, color='green', linestyle='--', label='Excellent (<0.5)')
axes[0].axhline(1.0, color='orange', linestyle='--', label='Good (<1.0)')
axes[0].set_xlabel('Image index')
axes[0].set_ylabel('Reprojection error (pixels)')
axes[0].set_title('Per-image reprojection error\n(green=excellent, orange=good, red=poor)')
axes[0].legend()

# Show error distribution
axes[1].hist(per_image_errors, bins=10, color='steelblue', edgecolor='white')
axes[1].axvline(np.mean(per_image_errors), color='red', linestyle='--',
                label=f'Mean: {np.mean(per_image_errors):.3f} px')
axes[1].set_xlabel('Reprojection error (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Error distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

worst_idx = np.argmax(per_image_errors)
print(f"Worst image: #{worst_idx} with error {per_image_errors[worst_idx]:.3f} px")
print("In practice: identify high-error images and recapture them")

## 3b. Diagnosing Calibration Quality Per Image

`calibrateCamera` gives you a single mean error for the whole calibration — but that average hides a lot. Some images might be contributing very little error while one or two "bad" images (blurry, partially visible, wrong corner count) are dragging the quality down.

The code below computes the reprojection error **for each image individually**, then plots two views of the same data.

---

### Graph 1 — Bar chart: per-image error

Each bar is one calibration image. The height is its individual reprojection error in pixels.

```
Image #0: 0.021 px  ← this image was clean, contributes little error
Image #1: 0.031 px
Image #2: 0.025 px  ← worst image (but still excellent here)
...
```

Colors tell you at a glance:
- **Green** (< 0.5 px) — excellent, keep it
- **Orange** (0.5–1.0 px) — acceptable, keep if you don't have better
- **Red** (> 1.0 px) — consider removing and recapturing

**How to use it:** spot any red or tall orange bars. Those are your weakest images. Remove them and re-run `calibrateCamera` — the mean error will likely improve.

---

### Graph 2 — Histogram: error distribution

A histogram answers a different question: **are all images roughly equally good, or are there outliers?**

Each image produces one reprojection error value (a continuous number like 0.023 px). The histogram groups those values into bins and shows how many images fall into each bin:

```
0.010–0.015 px: ██ (2 images)
0.015–0.020 px: ████ (4 images)
0.020–0.025 px: ████ (4 images)
0.025–0.030 px: ██ (2 images)
```

**How to read it:**
- **Narrow cluster** (all bars near each other) → consistent calibration, all images are similar quality — good
- **Wide spread or outlier bar far to the right** → some images are much worse than others — find and remove them

> **Honest note:** with only 12–15 calibration images the histogram isn't very informative — you don't have enough data points to see a meaningful distribution. The bar chart (Graph 1) is more useful at this scale. The histogram becomes genuinely helpful when you have 30+ images and need to quickly spot if a handful of outliers are skewing your mean error.

---

## 4. Saving and Loading Calibration Data

Always save your calibration results — calibrating for every session is wasteful. Three common formats:

| Format | Pros | Cons |
|---|---|---|
| `.npz` (NumPy) | Fast, compact, exact precision | NumPy-specific |
| `.json` (JSON) | Human-readable, language-agnostic | Slightly verbose |
| `.pickle` | Fast, Python-native | Not readable by other languages |

In [ ]:
# Save calibration results in multiple formats

save_dir = '../assets/calibration'

# --- Save as .npz (recommended for Python workflows) ---
npz_path = f'{save_dir}/camera_calibration.npz'
np.savez(npz_path,
         K=K,
         dist=dist,
         reprojection_error=ret,
         image_size=np.array(img_size))
print(f"Saved .npz: {npz_path}")

# --- Save as .json (for sharing / inspection) ---
json_path = f'{save_dir}/camera_calibration.json'
calib_data = {
    'reprojection_error': float(ret),
    'image_size': list(img_size),
    'camera_matrix': K.tolist(),
    'distortion_coefficients': dist.flatten().tolist(),
    'comments': 'Generated by notebook 07_camera_calibration.ipynb'
}
with open(json_path, 'w') as f:
    json.dump(calib_data, f, indent=2)
print(f"Saved .json: {json_path}")

# --- Load calibration back ---
print("\nLoading back from .npz:")
data = np.load(npz_path)
K_loaded    = data['K']
dist_loaded = data['dist']
err_loaded  = float(data['reprojection_error'])

print(f"  Reprojection error: {err_loaded:.4f} px")
print(f"  K matches original: {np.allclose(K_loaded, K)}")
print(f"  dist matches:       {np.allclose(dist_loaded, dist)}")

print("\nLoading back from .json:")
with open(json_path) as f:
    jd = json.load(f)
K_json = np.array(jd['camera_matrix'])
dist_json = np.array(jd['distortion_coefficients'])
print(f"  fx = {K_json[0,0]:.2f}, fy = {K_json[1,1]:.2f}, cx = {K_json[0,2]:.2f}, cy = {K_json[1,2]:.2f}")

## 5. Calibration Best Practices

Summarized from video notes 3, 4, and 15:

### Image capture tips

| Do | Don't |
|---|---|
| Take 10–20 images | Use fewer than 6 (solver degenerates) |
| Vary tilt: 30°, 45°, upside-down | Keep chessboard mostly flat |
| Cover all corners of the image frame | Cluster in the center only |
| Use different distances | Use one fixed distance |
| Use a rigid, flat board (foam board) | Use paper that bends |
| Check for blur before saving | Include blurry images |

### If reprojection error is high

1. Delete images with `per_image_errors > 1.5 px` and re-run
2. Capture more diverse angles
3. Make sure the chessboard is fully visible in every image
4. Use a physical board — printing on a phone screen introduces barrel distortion
5. If using a zoom lens, calibrate at each zoom level separately

### The < 5 minute workflow (video note 15)

1. Open webcam in Python
2. Press **S** to save a frame whenever the chessboard looks good
3. Take 15 images in ~2 minutes (vary angles)
4. Run `calibrateCamera` → save `.npz`
5. Done — use the same calibration for weeks until you change the lens or resolution

In [ ]:
# The live capture script lives in scripts/calibration/capture_calibration_images.py
# Run it from the repo root in a terminal:
#
#   python scripts/calibration/capture_calibration_images.py
#
# Optional arguments:
#   --save-dir  PATH    where to save images (default: calibration_images/)
#   --cols      INT     inner corner columns (default: 9)
#   --rows      INT     inner corner rows    (default: 6)
#   --camera    INT     webcam index         (default: 0)
#
# Example — custom board and save folder:
#   python scripts/calibration/capture_calibration_images.py --cols 9 --rows 6 --save-dir my_calib_imgs
#
# Controls while running:
#   S   → save frame (only when chessboard is detected)
#   Q   → quit
#
# Aim for 15+ images at varied angles, distances, and positions.
# Then point calibrateCamera at the saved folder.

print("See scripts/calibration/capture_calibration_images.py — run from repo root in a terminal.")

## Recap

| Step | Function | Output |
|---|---|---|
| Prepare board | Manual / print | `chessboard_9x6.png` |
| Detect corners | `findChessboardCorners` | `corners` (N×1×2) |
| Refine corners | `cornerSubPix` | Subpixel accurate corners |
| Calibrate | `calibrateCamera` | K, dist, rvecs, tvecs, error |
| Evaluate | Per-image reprojection error | < 0.5px = excellent |
| Save | `np.savez` or JSON | `camera_calibration.npz` |
| Load | `np.load` | K and dist for other notebooks |

---

**Next:** `08_distortion_undistortion.ipynb` — use K and dist to remove lens distortion from images.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Effect of image count on calibration error
# ============================================================
# Using the images already captured, run calibration with:
#   - 3 images, 5 images, 8 images, all images
# Plot how reprojection error changes as image count increases.
# What is the minimum number that gives < 1.0 px error?

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# counts = [3, 5, 8, len(objpoints)]
# errors = []
# for n in counts:
#     ret_n, _, _, _, _ = cv2.calibrateCamera(
#         objpoints[:n], imgpoints[:n], img_size, None, None)
#     errors.append(ret_n)
#     print(f"  {n} images → error = {ret_n:.4f} px")
#
# plt.figure(figsize=(8, 4))
# plt.plot(counts, errors, 'bo-', markersize=8)
# plt.axhline(1.0, color='orange', linestyle='--', label='Good threshold (1.0 px)')
# plt.axhline(0.5, color='green', linestyle='--', label='Excellent threshold (0.5 px)')
# plt.xlabel('Number of calibration images')
# plt.ylabel('Mean reprojection error (pixels)')
# plt.title('Calibration error vs image count')
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# ============================================================
# EXERCISE 2: Inspect the calibration JSON
# ============================================================
# Load the camera_calibration.json file we saved.
# Print a nicely formatted summary with:
#   - fx, fy (with units)
#   - cx, cy (with units)
#   - FOV horizontal and vertical (computed from K and image size)
#   - All 5 distortion coefficients with their names (k1, k2, p1, p2, k3)
#   - Quality assessment (excellent/good/poor)

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# with open('../assets/calibration/camera_calibration.json') as f:
#     d = json.load(f)
#
# K_j = np.array(d['camera_matrix'])
# dc  = d['distortion_coefficients']
# W, H = d['image_size']
# fov_x = 2 * np.degrees(np.arctan(W / (2 * K_j[0,0])))
# fov_y = 2 * np.degrees(np.arctan(H / (2 * K_j[1,1])))
# err = d['reprojection_error']
# q = 'Excellent' if err < 0.5 else 'Good' if err < 1.0 else 'Acceptable' if err < 2.0 else 'Poor'
#
# print("Camera Calibration Summary")
# print("="*40)
# print(f"  fx = {K_j[0,0]:.2f} px  |  fy = {K_j[1,1]:.2f} px")
# print(f"  cx = {K_j[0,2]:.2f} px  |  cy = {K_j[1,2]:.2f} px")
# print(f"  FOV: {fov_x:.1f}° (H) × {fov_y:.1f}° (V)")
# print(f"  Distortion: k1={dc[0]:.4f}, k2={dc[1]:.4f}, p1={dc[2]:.4f}, p2={dc[3]:.4f}, k3={dc[4]:.4f}")
# print(f"  Reprojection error: {err:.4f} px → {q}")